<a href="https://colab.research.google.com/github/OlatokeJumai/AML_CFT-Compliance/blob/main/UNILAG_KRR_XAI_AML_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Provenance-aware KRR–XAI for synthetic AML monitoring
## UNILAG MIT dissertation — Jumai Olatoke

Version 1.0 • 15 October 2026 • prospective research implementation

**Run all cells in order in Google Colab.** Default `MODE="SMOKE"` uses deliberately artificial fixture data only. Its numbers are NOT dissertation results. Set `MODE="RESEARCH"` only after uploading the unmodified IBM HI-Small transaction CSV and reviewing the configuration.

This notebook implements a detector, SHAP/LIME evaluation, structural rules, provenance records and controlled integrity tests. It does not certify CBN compliance, demographic fairness, user comprehension or Nigerian operational effectiveness. Developer tests are not independent validation.

No credentials are included. Revoke the Kaggle credential previously shared. Do not paste replacement secrets into this notebook.

Research sources: Altman et al. (2023), *Realistic Synthetic Financial Transactions for Anti-Money Laundering Models*, NeurIPS 36, 29851–29874; Nauta et al. (2023), doi:10.1145/3583558; CBN (2026), circular BSD/DIR/PUB/LAB/019/002; NIST (2023), doi:10.6028/NIST.AI.100-1. Dataset page: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml . Data licence must be checked separately from repository code licence.

## 0. Install dependencies
Internet is required in Colab for this cell, not for later analysis of an uploaded CSV. The exact resolved versions are exported. The environment file supports replay but does not prove bitwise reproducibility across different hardware. Restart the runtime if Colab requests it.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26,<3", "pandas>=2.2,<3", "scikit-learn>=1.4,<2",
    "xgboost>=2.1,<4", "shap>=0.46,<1", "lime==0.2.0.1", "scipy>=1.11,<2",
    "joblib>=1.3,<2", "matplotlib>=3.8,<4"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy>=1.26,<3', 'pandas>=2.2,<3', 'scikit-learn>=1.4,<2', 'xgboost>=2.1,<4', 'shap>=0.46,<1', 'lime==0.2.0.1', 'scipy>=1.11,<2', 'joblib>=1.3,<2', 'matplotlib>=3.8,<4'], returncode=0)

## 1. Frozen configuration
The full Chapter Three protocol is computationally demanding. In RESEARCH mode, defaults preserve the three-seed design and ten LIME repetitions. There is no automatic downsampling or silent fallback. Reducing settings requires a documented deviation. Feature construction can take substantial time and RAM; GPU acceleration does not accelerate the Python history pass.

The amount tiers, model tuning, encoding and threshold selection use training/validation only. Test labels are used for final evaluation and pre-specified diagnostic sampling, never for features or rules.

In [2]:
from pathlib import Path
import os, sys, json, hashlib, platform, time, copy, warnings
from collections import defaultdict, Counter, deque
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix, log_loss)
from sklearn.utils.class_weight import compute_sample_weight
import joblib

MODE = "RESEARCH"  # Change explicitly to RESEARCH after reviewing the notebook.
DATA_PATH = Path("/content/HI-Small_Trans.csv")
RUN_ROOT = Path("/content/aml_runs") if Path("/content").exists() else Path("aml_runs")
CONFIG = {
    "mode": MODE, "seeds": [42, 52, 62], "train_fraction": .60, "validation_fraction": .20,
    "window_hours": 24, "counterparty_threshold": 5,
    "global_shap_n": 2000, "diagnostic_per_cell": 25, "lime_repeats": 10,
    "lime_perturbations": 2000, "lime_reference_n": 10000,
    "max_estimators": 300, "early_stopping_rounds": 30,
    "depths": [3, 6], "learning_rates": [.05, .10], "weighting": ["none", "balanced"],
    "primary_seed": 42, "top_k": 5, "sparse_positive_cutoff": 20,
    "sensitivity_hours": [6, 24], "sensitivity_thresholds": [3, 5, 10],
    "feature_version": "history-v1", "rule_version": "structural-v1",
    "self_transfer_policy": "exclude from relational histories and rule matches",
    "deviations": []
}
if MODE == "SMOKE":
    CONFIG.update(seeds=[42], global_shap_n=60, diagnostic_per_cell=2, lime_repeats=2,
        lime_perturbations=150, lime_reference_n=300, max_estimators=35,
        early_stopping_rounds=8, depths=[3], learning_rates=[.10], weighting=["none"])
    CONFIG["deviations"] = ["SMOKE ONLY: artificial fixtures and reduced compute; not research evidence."]
assert MODE in {"SMOKE", "RESEARCH"}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = RUN_ROOT / (MODE.lower() + "_" + RUN_ID)
RUN_DIR.mkdir(parents=True, exist_ok=False)

def plain(v):
    if isinstance(v, dict): return {str(k):plain(x) for k,x in v.items()}
    if isinstance(v, (list,tuple)): return [plain(x) for x in v]
    if isinstance(v, (np.integer,np.bool_)): return v.item()
    if isinstance(v, (np.floating,float)): return float(v) if np.isfinite(v) else None
    if isinstance(v, (pd.Timestamp,datetime)): return v.isoformat()
    if isinstance(v, Path): return str(v)
    return v

def write_json(name, value):
    (RUN_DIR/name).write_text(json.dumps(plain(value), indent=2, allow_nan=False), encoding="utf-8")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""): h.update(chunk)
    return h.hexdigest()

write_json("configuration.json", CONFIG)
print(MODE, RUN_DIR)

RESEARCH /content/aml_runs/research_20261015T165721_873858Z


## 2. Dataset acquisition and verified loading
In RESEARCH mode upload **only `HI-Small_Trans.csv`** to `/content/data/` using the Colab Files sidebar, or change `DATA_PATH` to its Drive location. Download the named file from the official dataset page; do not download the entire multi-variant archive unnecessarily. Pattern annotation files are not parsed by this version and must not be claimed as evaluated.

Bank/account and currency strings are preserved. Invalid essential fields are quarantined rather than silently repaired. Negative and non-finite amounts are quarantined. Exact duplicates are reported and retained: they are not automatically errors. The source file is fingerprinted before analysis.

In [3]:
def make_fixture(n=1800, seed=42):
    r=np.random.default_rng(seed)
    t=pd.Timestamp("2022-09-01")+pd.to_timedelta(np.arange(n)*10,unit="min")
    src=r.integers(0,70,n); dst=r.integers(0,70,n)
    amount=r.lognormal(5,1.2,n)
    label=((amount>450)&(src%3==0)).astype(int)
    label[np.arange(0,n,59)]=1
    return pd.DataFrame({"Timestamp":t.astype(str),"From Bank":"1","Account":src.astype(str),
        "To Bank":"1","Account.1":dst.astype(str),"Amount Received":amount,
        "Receiving Currency":"USD","Amount Paid":amount,"Payment Currency":"USD",
        "Payment Format":r.choice(["ACH","Wire","Cheque"],n),"Is Laundering":label})

if MODE == "RESEARCH":
    if not DATA_PATH.is_file():
        raise FileNotFoundError("Upload the original HI-Small_Trans.csv and set DATA_PATH, then rerun this cell.")
    source_hash=sha256_file(DATA_PATH)
    raw=pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
    source_name=DATA_PATH.name
else:
    raw=make_fixture(); source_name="ARTIFICIAL_SMOKE_FIXTURE"
    source_hash=hashlib.sha256(raw.to_csv(index=False).encode()).hexdigest()

required=["Timestamp","From Bank","Account","To Bank","Account.1","Amount Received",
          "Receiving Currency","Amount Paid","Payment Currency","Payment Format","Is Laundering"]
missing=set(required)-set(raw.columns)
if missing: raise ValueError(f"Unexpected schema; missing {sorted(missing)}. Inspect original headers; do not guess mappings.")
rename={"Timestamp":"timestamp","From Bank":"src_bank","Account":"src_account",
    "To Bank":"dst_bank","Account.1":"dst_account","Amount Received":"received",
    "Receiving Currency":"recv_currency","Amount Paid":"paid","Payment Currency":"pay_currency",
    "Payment Format":"payment_format","Is Laundering":"label"}
d=raw[required].rename(columns=rename).copy(); d["row_id"]=np.arange(len(d),dtype=np.int64)
d["timestamp"]=pd.to_datetime(d.timestamp, errors="coerce")
for c in ["paid","received","label"]: d[c]=pd.to_numeric(d[c],errors="coerce")
reasons=pd.Series("",index=d.index)
def mark(mask, reason):
    global reasons
    reasons.loc[mask] += reason+";"
mark(d.timestamp.isna(),"invalid_timestamp")
for c in ["src_bank","src_account","dst_bank","dst_account","pay_currency","recv_currency","payment_format"]:
    d[c]=d[c].astype(str).str.strip()
    mark(d[c].isin(["","nan","None"]),"missing_"+c)
for c in ["paid","received"]:
    mark(~np.isfinite(d[c]) | (d[c]<0),"invalid_"+c)
mark(~d.label.isin([0,1]),"invalid_label")
bad=reasons.ne("")
quarantine=d.loc[bad].copy(); quarantine["reason"]=reasons[bad]
quarantine.to_csv(RUN_DIR/"quarantine.csv",index=False)
d=d.loc[~bad].sort_values(["timestamp","row_id"],kind="stable").reset_index(drop=True)
d["label"]=d.label.astype(np.int8)
# Factorisation uses composite bank-account tuples across both ends.
all_accounts=pd.concat([d[["src_bank","src_account"]].set_axis(["bank","account"],axis=1),
                        d[["dst_bank","dst_account"]].set_axis(["bank","account"],axis=1)],ignore_index=True)
ids,_=pd.factorize(pd.MultiIndex.from_frame(all_accounts),sort=False)
d["src"]=ids[:len(d)].astype(np.int32); d["dst"]=ids[len(d):].astype(np.int32)
d["tsec"]=d.timestamp.astype("int64")//10**9
d["self_transfer"]=(d.src==d.dst).astype(np.int8)
assert len(d)>100 and d.label.nunique()==2
manifest={"source":source_name,"sha256":source_hash,"mode":MODE,"raw_rows":len(raw),
    "accepted_rows":len(d),"quarantined_rows":int(bad.sum()),"columns":list(raw.columns),
    "time_min":d.timestamp.min(),"time_max":d.timestamp.max(),"label_counts":d.label.value_counts().to_dict(),
    "exact_duplicates_retained":int(raw.duplicated().sum()),
    "payment_formats":d.payment_format.value_counts().to_dict(),
    "payment_currencies":d.pay_currency.value_counts().to_dict(),
    "pattern_annotations":"NOT_EVALUATED","real_customer_data":False}
write_json("dataset_manifest.json",manifest)
print(json.dumps(plain(manifest),indent=2))
del raw, all_accounts, ids, quarantine

{
  "source": "HI-Small_Trans.csv",
  "sha256": "9effb24c3e080db4c09719c3a2bcec8134b4de018128143744d5b5e621338e1f",
  "mode": "RESEARCH",
  "raw_rows": 311725,
  "accepted_rows": 311724,
  "quarantined_rows": 1,
  "columns": [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
  ],
  "time_min": "2022-09-01T00:00:00",
  "time_max": "2022-09-04T15:48:00",
  "label_counts": {
    "0": 311696,
    "1": 28
  },
  "exact_duplicates_retained": 0,
  "payment_formats": {
    "Reinvestment": 225763,
    "Cheque": 31196,
    "Credit Card": 28139,
    "ACH": 14574,
    "Cash": 8321,
    "Wire": 3657,
    "Bitcoin": 74
  },
  "payment_currencies": {
    "US Dollar": 122424,
    "Euro": 75509,
    "Swiss Franc": 15255,
    "Yuan": 13694,
    "Rupee": 12581,
    "Shekel": 11123,
    "UK Pound": 11123,
    "Yen": 10323,
    "Ruble": 10060,

## 3. Streaming history features
Strict window: prior timestamps in `(current_time - window, current_time)`. Same-timestamp transactions are featurised before ANY member of that batch enters history. Self-transfers are excluded from relational histories and candidate rules, not from classifier evaluation. Currency totals are separated; there is no implicit FX conversion. History completeness is global observation coverage, not proof of account-opening history.

An O(n) Python history pass maintains active-window counters and exact observed pairs. Large files can require a high-memory runtime. This cell never uses labels. Counterparty identifiers are retained internally for histories but not supplied as predictive columns.

In [4]:
def build_history(data, hours=24):
    window=int(hours*3600); n=len(data)
    cols=["sender_out_count","recipient_in_count","sender_distinct_recipients",
          "recipient_distinct_senders","sender_distinct_senders","sender_out_amount",
          "sender_in_amount","recipient_in_amount","seconds_since_sender","new_pair",
          "sender_seen","window_complete"]
    out=np.zeros((n,len(cols)),dtype=np.float64)
    outgoing={}; incoming={}; sums_out=defaultdict(float); sums_in=defaultdict(float)
    last={}; seen_pairs=set(); active=deque(); first=int(data.tsec.iloc[0])
    times=data.tsec.to_numpy(); starts=np.r_[0,np.flatnonzero(times[1:]!=times[:-1])+1,n]
    records=list(zip(data.src.to_numpy(),data.dst.to_numpy(),data.pay_currency.to_numpy(),
                     data.recv_currency.to_numpy(),data.paid.to_numpy(),data.received.to_numpy()))
    def inc(store,a,b,delta):
        if delta>0:
            if a not in store: store[a]=Counter()
            store[a][b]+=1
        else:
            store[a][b]-=1
            if store[a][b]==0: del store[a][b]
            if not store[a]: del store[a]
    for lo,hi in zip(starts[:-1],starts[1:]):
        now=int(times[lo]); cutoff=now-window
        while active and active[0][0]<=cutoff:
            _,s,t,pc,rc,p,r=active.popleft()
            inc(outgoing,s,t,-1); inc(incoming,t,s,-1)
            sums_out[(s,pc)]-=p; sums_in[(t,rc)]-=r
            # Remove dead keys without retaining near-zero floating residues.
            if s not in outgoing:
                for key in [k for k in sums_out if k[0]==s]: del sums_out[key]
            if t not in incoming:
                for key in [k for k in sums_in if k[0]==t]: del sums_in[key]
        for i in range(lo,hi):
            s,t,pc,rc,p,r=records[i]
            oc=outgoing.get(s,{}); ic=incoming.get(t,{})
            out[i]=[sum(oc.values()),sum(ic.values()),len(oc),len(ic),len(incoming.get(s,{})),
                max(0,sums_out.get((s,pc),0)),max(0,sums_in.get((s,pc),0)),
                max(0,sums_in.get((t,rc),0)),now-last[s] if s in last else 0,
                int((s,t) not in seen_pairs),int(s in last),int(now-first>=window)]
        for i in range(lo,hi):
            s,t,pc,rc,p,r=records[i]
            if s==t: continue
            inc(outgoing,s,t,1); inc(incoming,t,s,1)
            sums_out[(s,pc)]+=p; sums_in[(t,rc)]+=r
            last[s]=now; seen_pairs.add((s,t)); active.append((now,s,t,pc,rc,p,r))
    result=pd.DataFrame(out,columns=cols,index=data.index)
    # Inclusive distinct counts add the current edge once, not all simultaneous edges.
    # These use a separate deterministic pass with the same expiry policy below.
    return result

def inclusive_counts(data,hours=24):
    window=int(hours*3600); active=deque(); outgoing={}; incoming={}
    res=np.zeros((len(data),3),dtype=np.int32)
    times=data.tsec.to_numpy(); starts=np.r_[0,np.flatnonzero(times[1:]!=times[:-1])+1,len(data)]
    ss=data.src.to_numpy(); tt=data.dst.to_numpy()
    def change(store,a,b,delta):
        if delta>0:
            store.setdefault(a,Counter())[b]+=1
        else:
            store[a][b]-=1
            if store[a][b]==0: del store[a][b]
            if not store[a]: del store[a]
    for lo,hi in zip(starts[:-1],starts[1:]):
        now=int(times[lo])
        while active and active[0][0]<=now-window:
            _,s,t=active.popleft(); change(outgoing,s,t,-1); change(incoming,t,s,-1)
        for i in range(lo,hi):
            s,t=int(ss[i]),int(tt[i]); oc=outgoing.get(s,{}); ic=incoming.get(t,{})
            res[i]=[len(ic)+int(s!=t and s not in ic),len(oc)+int(s!=t and t not in oc),len(incoming.get(s,{}))]
        for i in range(lo,hi):
            s,t=int(ss[i]),int(tt[i])
            if s!=t:
                change(outgoing,s,t,1); change(incoming,t,s,1); active.append((now,s,t))
    return pd.DataFrame(res,columns=["recipient_senders_inclusive","sender_recipients_inclusive","sender_senders_prior"],index=data.index)

t0=time.perf_counter()
history=build_history(d,CONFIG["window_hours"])
rule_counts=inclusive_counts(d,CONFIG["window_hours"])
feature_seconds=time.perf_counter()-t0
print({"history_seconds":feature_seconds,"rows":len(history)})

{'history_seconds': 409.01951088299995, 'rows': 311724}


## 4. Timestamp-grouped partitions and feature matrices
Bank/account IDs, original row IDs, labels, and annotation files never enter predictors. The feature pass is online across validation/test timestamps without using their labels. Future transactions cannot affect earlier history. Model fitting and transformations still use training data only.

In [5]:
t1=d.timestamp.iloc[int(len(d)*CONFIG["train_fraction"])]
t2=d.timestamp.iloc[int(len(d)*(CONFIG["train_fraction"]+CONFIG["validation_fraction"]))]
train=(d.timestamp<t1).to_numpy(); valid=((d.timestamp>=t1)&(d.timestamp<t2)).to_numpy(); test=(d.timestamp>=t2).to_numpy()
assert all(mask.sum()>0 for mask in [train,valid,test])
assert not np.any(train&valid) and not np.any(valid&test)
assert d.loc[train,"timestamp"].max()<d.loc[valid,"timestamp"].min()
assert d.loc[valid,"timestamp"].max()<d.loc[test,"timestamp"].min()
for name,mask in [("train",train),("validation",valid),("test",test)]:
    if d.loc[mask,"label"].nunique()!=2:
        raise ValueError(f"{name} has only one label. Report and redesign explicitly; no silent split change.")
base=pd.DataFrame({"log_paid":np.log1p(d.paid),"log_received":np.log1p(d.received),
    "payment_format":d.payment_format,"pay_currency":d.pay_currency,"recv_currency":d.recv_currency,
    "self_transfer":d.self_transfer})
full=pd.concat([base,history],axis=1)
# Heavy-tailed totals and elapsed times receive predefined log transforms.
for c in ["sender_out_amount","sender_in_amount","recipient_in_amount","seconds_since_sender"]:
    full[c]=np.log1p(full[c].clip(lower=0))
cat_cols=["payment_format","pay_currency","recv_currency"]
y=d.label.to_numpy()
split_ids=pd.DataFrame({"row_id":d.row_id,"timestamp":d.timestamp,
    "partition":np.select([train,valid,test],["train","validation","test"],default="ERROR")})
split_ids.to_csv(RUN_DIR/"split_membership.csv.gz",index=False,compression="gzip")
write_json("split_manifest.json",{name:{"rows":int(mask.sum()),"positives":int(y[mask].sum()),
    "start":d.loc[mask,"timestamp"].min(),"end":d.loc[mask,"timestamp"].max()}
    for name,mask in [("train",train),("validation",valid),("test",test)]})
write_json("feature_schema.json",{"base_columns":list(base),"history_columns":list(full),
    "excluded":["label","row_id","src","dst","bank and account strings","pattern annotations"],
    "history_seconds":feature_seconds,"same_timestamp_policy":"strictly earlier batches only"})
print(pd.crosstab(split_ids.partition,d.label))

label            0   1
partition             
test         62561  16
train       186930   8
validation   62205   4


## 5. Model tuning — validation only
Candidate settings are fixed in configuration. XGBoost early stopping uses validation data, never test data. `scale_pos_weight` is evaluated without SMOTE. The primary enhanced model is the seed-42 model chosen by validation average precision. Repeated-seed test results are not used to select the primary model.

For cost transparency, the full protocol fits both tabular and history-enriched XGBoost grids. No research result is generated until this cell completes.

In [6]:
from xgboost import XGBClassifier

def make_preprocessor(frame, scale=False):
    numeric=[c for c in frame if c not in cat_cols]
    return ColumnTransformer([
        ("num",StandardScaler() if scale else "passthrough",numeric),
        ("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=True),cat_cols)],
        sparse_threshold=1.0)

def choose_threshold(yt,scores):
    p,r,th=precision_recall_curve(yt,scores)
    f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(p[:-1]),where=(p[:-1]+r[:-1])>0)
    best=np.nanmax(f)
    return float(th[np.flatnonzero(np.isclose(f,best,rtol=0,atol=1e-12))[-1]])

models={}; tuning=[]
pre_lr=make_preprocessor(base,scale=True)
lr_train=pre_lr.fit_transform(base.loc[train]); lr_val=pre_lr.transform(base.loc[valid])
lr=LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42,solver="saga",tol=1e-3)
t0=time.perf_counter(); lr.fit(lr_train,y[train]); lr_seconds=time.perf_counter()-t0
lr_scores=lr.predict_proba(lr_val)[:,1]
models["logistic_transaction"]={"model":lr,"pre":pre_lr,"columns":list(base),
    "threshold":choose_threshold(y[valid],lr_scores),"validation_ap":average_precision_score(y[valid],lr_scores),
    "fit_seconds":lr_seconds,"seed":42}
for tag,frame in [("xgb_transaction",base),("xgb_history",full)]:
    pre=make_preprocessor(frame); a=pre.fit_transform(frame.loc[train]); b=pre.transform(frame.loc[valid])
    best=None
    for depth in CONFIG["depths"]:
        for rate in CONFIG["learning_rates"]:
            for weighting in CONFIG["weighting"]:
                weight=float((y[train]==0).sum()/max(1,(y[train]==1).sum())) if weighting=="balanced" else 1.
                params=dict(n_estimators=CONFIG["max_estimators"],max_depth=depth,learning_rate=rate,
                    subsample=.8,colsample_bytree=.8,objective="binary:logistic",eval_metric="aucpr",
                    tree_method="hist",early_stopping_rounds=CONFIG["early_stopping_rounds"],
                    scale_pos_weight=weight,random_state=42,n_jobs=2)
                m=XGBClassifier(**params)
                t0=time.perf_counter(); m.fit(a,y[train],eval_set=[(b,y[valid])],verbose=False)
                seconds=time.perf_counter()-t0; vs=m.predict_proba(b)[:,1]; ap=average_precision_score(y[valid],vs)
                tuning.append(dict(model=tag,depth=depth,rate=rate,weighting=weighting,weight=weight,
                    validation_ap=ap,best_iteration=m.best_iteration,fit_seconds=seconds))
                if best is None or ap>best[0]: best=(ap,m,params,vs,seconds)
    ap,m,params,vs,seconds=best
    models[tag]={"model":m,"pre":pre,"columns":list(frame),"threshold":choose_threshold(y[valid],vs),
        "validation_ap":ap,"fit_seconds":seconds,"seed":42,"params":params}
    for seed in CONFIG["seeds"]:
        if seed==42: continue
        p=dict(params,random_state=seed); repeat=XGBClassifier(**p)
        t0=time.perf_counter(); repeat.fit(a,y[train],eval_set=[(b,y[valid])],verbose=False)
        repeat_vs=repeat.predict_proba(b)[:,1]
        models[f"{tag}_seed{seed}"]={"model":repeat,"pre":pre,"columns":list(frame),"seed":seed,
            "threshold":choose_threshold(y[valid],repeat_vs),"validation_ap":average_precision_score(y[valid],repeat_vs),
            "fit_seconds":time.perf_counter()-t0,"params":p}
    del a,b
pd.DataFrame(tuning).to_csv(RUN_DIR/"validation_tuning.csv",index=False)
for name,obj in models.items():
    joblib.dump(obj,RUN_DIR/f"model_{name}.joblib")
write_json("model_registry.json",{name:{k:v for k,v in obj.items() if k not in ["model","pre"]} for name,obj in models.items()})
primary=models["xgb_history"]
print("Models fitted; primary chosen on validation only.")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Models fitted; primary chosen on validation only.


## 6. Frozen test evaluation and diagnostic selection
This is the first use of test labels for numerical performance reporting. Seed replication describes training variability, not independent data replications. PR evaluation is **average precision**, not an unspecified trapezoidal PR-AUC. Undefined denominators are recorded as null, not misleading perfect scores.

In [7]:
def metrics(yt,scores,threshold):
    pred=(np.asarray(scores)>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(yt,pred,labels=[0,1]).ravel()
    return {"n":len(yt),"positives":int(np.sum(yt)),"threshold":threshold,"tn":int(tn),"fp":int(fp),
        "fn":int(fn),"tp":int(tp),"precision":tp/(tp+fp) if tp+fp else None,
        "recall":tp/(tp+fn) if tp+fn else None,"f1":2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,
        "average_precision":average_precision_score(yt,scores) if np.sum(yt)>0 else None,
        "roc_auc":roc_auc_score(yt,scores) if len(np.unique(yt))==2 else None,
        "alert_rate":float(pred.mean()),"fpr":fp/(fp+tn) if fp+tn else None,
        "fnr":fn/(fn+tp) if fn+tp else None}
results=[]; test_pos=np.flatnonzero(test)
for name,obj in models.items():
    frame=full[obj["columns"]]
    scores=obj["model"].predict_proba(obj["pre"].transform(frame.loc[test]))[:,1]
    for label,threshold in [("validation_f1",obj["threshold"]),("reference_050",.5)]:
        results.append(dict(model=name,seed=obj["seed"],threshold_policy=label,mode=MODE,
            **metrics(y[test],scores,threshold)))
    if name=="xgb_history": primary_scores=scores
pd.DataFrame(results).to_csv(RUN_DIR/"classification_results.csv",index=False)
pred=(primary_scores>=primary["threshold"]).astype(int)
category=np.select([(y[test]==1)&(pred==1),(y[test]==0)&(pred==1),(y[test]==1)&(pred==0)],
                   ["TP","FP","FN"],default="TN")
predictions=pd.DataFrame({"position":test_pos,"row_id":d.loc[test,"row_id"].to_numpy(),
    "timestamp":d.loc[test,"timestamp"].to_numpy(),"label":y[test],"score":primary_scores,
    "prediction":pred,"category":category,"mode":MODE})
predictions.to_csv(RUN_DIR/"test_predictions.csv.gz",index=False,compression="gzip")
rng=np.random.default_rng(42); diag=[]
for c in ["TP","FP","FN","TN"]:
    pool=predictions.loc[predictions.category==c,"position"].to_numpy()
    diag.extend(rng.choice(pool,min(len(pool),CONFIG["diagnostic_per_cell"]),replace=False).tolist())
diag=np.array(sorted(diag),dtype=int)
pop=np.sort(rng.choice(test_pos,min(CONFIG["global_shap_n"],len(test_pos)),replace=False))
pd.DataFrame({"position":diag,"row_id":d.loc[diag,"row_id"].to_numpy()}).to_csv(RUN_DIR/"diagnostic_sample.csv",index=False)
write_json("explanation_sample_manifest.json",{"population_n":len(pop),"population_positives":int(y[pop].sum()),
    "diagnostic_n":len(diag),"diagnostic_counts":predictions[predictions.position.isin(diag)].category.value_counts().to_dict(),
    "population_positions":pop.tolist(),"diagnostic_positions":diag.tolist(),
    "warning":"Diagnostic cases are stratified by outcome; not population estimates."})
print(pd.DataFrame(results)[["model","threshold_policy","precision","recall","f1","average_precision"]])

                     model threshold_policy  precision  recall        f1  \
0     logistic_transaction    validation_f1   0.001094  0.5625  0.002185   
1     logistic_transaction    reference_050   0.001056  0.5625  0.002109   
2          xgb_transaction    validation_f1   0.005236  0.4375  0.010347   
3          xgb_transaction    reference_050        NaN  0.0000  0.000000   
4   xgb_transaction_seed52    validation_f1   0.004213  0.1875  0.008242   
5   xgb_transaction_seed52    reference_050        NaN  0.0000  0.000000   
6   xgb_transaction_seed62    validation_f1   0.003584  0.1875  0.007034   
7   xgb_transaction_seed62    reference_050        NaN  0.0000  0.000000   
8              xgb_history    validation_f1   0.000000  0.0000  0.000000   
9              xgb_history    reference_050   0.006812  0.3125  0.013333   
10      xgb_history_seed52    validation_f1   0.000000  0.0000  0.000000   
11      xgb_history_seed52    reference_050   0.004023  0.3125  0.007943   
12      xgb_

## 7. Structural rule engine and deterministic unit tests
These are researcher-defined **candidate structures**, not validated FATF typology classifiers. Thresholds are not legal reporting limits. Multiple matches remain visible. No automated STR/SAR filing is recommended. A fully observed window plus valid IDs is required. In this release transaction schema validation occurs before rule evaluation; bad-ID records are quarantined.

The main rule cache is computed independently for all test transactions, including non-alerts. Fixture tests validate rule-code agreement with its specification, not real-world laundering validity.

In [8]:
RULE_SOURCES={"structural":"Altman et al. (2023), synthetic AML patterns; not a CBN threshold",
              "governance":"CBN (2026), section 5.9; traceability expectation, not rule approval"}

def evaluate_rules(counts,complete,self_transfer,k=5,hours=24):
    values={"recipient_senders_inclusive":int(counts[0]),"sender_recipients_inclusive":int(counts[1]),
            "sender_senders_prior":int(counts[2])}
    conditions={"R1":values["recipient_senders_inclusive"]>=k,
        "R2":values["sender_recipients_inclusive"]>=k,
        "R3":values["sender_senders_prior"]>=k and values["sender_recipients_inclusive"]>=k}
    result=[]
    for rid,name in [("R1","Recipient fan-in candidate"),("R2","Sender fan-out candidate"),("R3","Gather-scatter candidate")]:
        status="NOT_EVALUABLE" if not complete else ("NOT_MATCHED" if self_transfer else ("MATCHED" if conditions[rid] else "NOT_MATCHED"))
        reason="incomplete_global_observation_window" if not complete else ("self_transfer_excluded" if self_transfer else None)
        result.append({"rule_id":rid,"name":name,"version":CONFIG["rule_version"],"status":status,
            "reason":reason,"observed_counts":values,"window_hours":hours,"counterparty_threshold":k,
            "source":RULE_SOURCES["structural"],"parameter_authority":"researcher-defined",
            "conclusion_limit":"Candidate structure only; not proof of an offence or classifier reasoning."})
    return result

def run_rule_tests():
    cases=[("below",[4,4,4],True,False,["NOT_MATCHED"]*3),
        ("exact",[5,5,5],True,False,["MATCHED"]*3),
        ("fan_in_only",[5,1,1],True,False,["MATCHED","NOT_MATCHED","NOT_MATCHED"]),
        ("warmup",[10,10,10],False,False,["NOT_EVALUABLE"]*3),
        ("self",[10,10,10],True,True,["NOT_MATCHED"]*3)]
    rows=[]
    for name,c,complete,self_,expected in cases:
        got=[x["status"] for x in evaluate_rules(c,complete,self_)]
        rows.append({"test":name,"passed":got==expected,"expected":str(expected),"actual":str(got)})
    assert all(x["passed"] for x in rows)
    return rows

unit_results=run_rule_tests()
pd.DataFrame(unit_results).to_csv(RUN_DIR/"rule_unit_tests.csv",index=False)
main_counts=rule_counts.to_numpy()
def rules_at(pos,k=None,hours=None,counts=None):
    hh=CONFIG["window_hours"] if hours is None else hours
    complete=(int(d.tsec.iloc[pos])-int(d.tsec.iloc[0]))>=hh*3600
    return evaluate_rules(main_counts[pos] if counts is None else counts,complete,bool(d.self_transfer.iloc[pos]),
        CONFIG["counterparty_threshold"] if k is None else k,hh)
coverage=[]; sensitivity=[]
for hours in CONFIG["sensitivity_hours"]:
    cnt=main_counts if hours==CONFIG["window_hours"] else inclusive_counts(d,hours).to_numpy()
    complete=(d.tsec.to_numpy()-int(d.tsec.iloc[0]))>=hours*3600
    evaluable=complete[test]; self_=d.self_transfer.to_numpy()[test].astype(bool)
    c=cnt[test]
    for k in CONFIG["sensitivity_thresholds"]:
        matches=[(c[:,0]>=k),(c[:,1]>=k),(c[:,2]>=k)&(c[:,1]>=k)]
        for j,match in enumerate(matches,1):
            match=match&evaluable&~self_
            for label in [0,1]:
                mask=y[test]==label
                row={"hours":hours,"k":k,"rule":f"R{j}","label":label,
                    "total":int(mask.sum()),"evaluable":int((mask&evaluable).sum()),
                    "matched":int((mask&match).sum()),"not_evaluable":int((mask&~evaluable).sum()),"mode":MODE}
                sensitivity.append(row)
                if hours==CONFIG["window_hours"] and k==CONFIG["counterparty_threshold"]: coverage.append(row)
pd.DataFrame(coverage).to_csv(RUN_DIR/"rule_coverage.csv",index=False)
pd.DataFrame(sensitivity).to_csv(RUN_DIR/"rule_sensitivity.csv",index=False)
print(pd.DataFrame(coverage))

   hours  k rule  label  total  evaluable  matched  not_evaluable      mode
0     24  5   R1      0  62561          3        0          62558  RESEARCH
1     24  5   R1      1     16          9        0              7  RESEARCH
2     24  5   R2      0  62561          3        0          62558  RESEARCH
3     24  5   R2      1     16          9        1              7  RESEARCH
4     24  5   R3      0  62561          3        0          62558  RESEARCH
5     24  5   R3      1     16          9        0              7  RESEARCH


## 8. SHAP attribution — raw margin, not probability
Full vectors are saved for both samples. Additivity is checked on the raw model margin; a small residual is not proof of causal validity or complete fidelity. The selected early-stopping tree range is fixed explicitly so predictor and explainer use the same model.

In [9]:
import shap
from scipy import sparse
primary_model=primary["model"]
# Slice the booster to the selected early-stopping iteration.
booster=primary_model.get_booster()[:primary_model.best_iteration+1]
pre=primary["pre"]
encoded_names=list(pre.get_feature_names_out())
explainer=shap.TreeExplainer(booster,model_output="raw",feature_perturbation="tree_path_dependent")
def encoded_dense(positions):
    z=pre.transform(full.iloc[positions][primary["columns"]])
    return z.toarray() if sparse.issparse(z) else np.asarray(z)

def get_shap(positions,tag):
    z=encoded_dense(positions)
    start=time.perf_counter(); vals=np.asarray(explainer.shap_values(z)); seconds=time.perf_counter()-start
    import xgboost as xgb
    margin=booster.predict(xgb.DMatrix(z,feature_names=booster.feature_names),output_margin=True)
    baseline=float(np.asarray(explainer.expected_value).reshape(-1)[0])
    residual=np.abs(baseline+vals.sum(axis=1)-margin)
    np.savez_compressed(RUN_DIR/f"shap_{tag}.npz",values=vals,positions=positions,
        feature_names=np.array(encoded_names),baseline=baseline,margin=margin,residual=residual)
    return vals,{"sample":tag,"n":len(positions),"seconds":seconds,"output_scale":"raw_margin",
        "baseline":baseline,"mean_additivity_abs_error":float(residual.mean()),"max_additivity_abs_error":float(residual.max())}
shap_pop,sp=get_shap(pop,"population")
shap_diag,sd=get_shap(diag,"diagnostic")
pd.DataFrame([sp,sd]).to_csv(RUN_DIR/"shap_checks.csv",index=False)
pd.DataFrame({"encoded_feature":encoded_names,"mean_abs_shap":np.abs(shap_pop).mean(axis=0)}).sort_values(
    "mean_abs_shap",ascending=False).to_csv(RUN_DIR/"shap_population_importance.csv",index=False)
print(pd.DataFrame([sp,sd]))

       sample     n   seconds output_scale  baseline  \
0  population  2000  0.016967   raw_margin  0.199332   
1  diagnostic    46  0.001388   raw_margin  0.199332   

   mean_additivity_abs_error  max_additivity_abs_error  
0               1.269579e-08              2.384186e-07  
1               2.818263e-08              1.937151e-07  


## 9. LIME — original categorical fields, local agreement and repeatability
LIME is fit on the primary feature frame before one-hot encoding so categorical perturbations remain categories. Model predictions pass through the frozen preprocessor. Historical aggregate features can still be perturbed into financially implausible combinations: report this limitation. No synthetic LIME neighbourhood is claimed to be a real transaction distribution.

Pairwise results report top-k overlap AND ranking agreement; disappearing features are not hidden by reporting only common-feature Kendall tau. The top-feature sign agreement denominator is the union of selected features, penalising missing features. No arbitrary pass/fail threshold is assigned.

In [10]:
from lime.lime_tabular import LimeTabularExplainer
lime_columns=primary["columns"]
codebooks={c:sorted(full.loc[train,c].astype(str).unique().tolist())+["__UNKNOWN__"] for c in cat_cols}
cat_indices=[lime_columns.index(c) for c in cat_cols]

def to_lime(frame):
    z=frame[lime_columns].copy()
    for c in cat_cols:
        mapping={v:i for i,v in enumerate(codebooks[c])}
        z[c]=z[c].astype(str).map(mapping).fillna(len(codebooks[c])-1)
    return z.to_numpy(dtype=float)

def lime_predict(z):
    frame=pd.DataFrame(z,columns=lime_columns)
    for c in cat_cols:
        codes=np.rint(frame[c]).astype(int).clip(0,len(codebooks[c])-1)
        frame[c]=[codebooks[c][v] for v in codes]
    return primary_model.predict_proba(pre.transform(frame[lime_columns]))

ref_positions=np.random.default_rng(42).choice(np.flatnonzero(train),min(CONFIG["lime_reference_n"],train.sum()),replace=False)
ref=to_lime(full.iloc[ref_positions])
diag_lime=to_lime(full.iloc[diag]); lime_rows=[]; lime_pairs=[]; lime_first={}
for pos,row in zip(diag,diag_lime):
    maps=[]
    for run in range(CONFIG["lime_repeats"]):
        le=LimeTabularExplainer(ref,feature_names=lime_columns,class_names=["normal","laundering-labelled"],
            categorical_features=cat_indices,categorical_names={lime_columns.index(c):codebooks[c] for c in cat_cols},
            mode="classification",random_state=4200+run,discretize_continuous=True)
        start=time.perf_counter()
        ex=le.explain_instance(row,lime_predict,labels=(1,),num_features=min(CONFIG["top_k"],len(row)),
            num_samples=CONFIG["lime_perturbations"])
        seconds=time.perf_counter()-start
        pairs=sorted(ex.as_map()[1],key=lambda z:abs(z[1]),reverse=True)
        weights={int(i):float(w) for i,w in pairs}; maps.append(weights)
        score=float(lime_predict(row.reshape(1,-1))[0,1]); local=float(np.asarray(ex.local_pred).ravel()[0])
        record={"position":int(pos),"run":run,"seed":4200+run,"fit_r2":float(ex.score),
            "model_score":score,"surrogate_local_prediction":local,"local_abs_error":abs(score-local),
            "seconds":seconds,"features":{lime_columns[i]:w for i,w in weights.items()},
            "descriptions":ex.as_list(label=1)}
        lime_rows.append(record)
        if run==0: lime_first[int(pos)]=record
    for i in range(len(maps)):
        for j in range(i+1,len(maps)):
            a,b=maps[i],maps[j]; aa=list(a); bb=list(b); common=set(a)&set(b); union=set(a)|set(b)
            tau=kendalltau([aa.index(k) for k in sorted(common)],[bb.index(k) for k in sorted(common)]).statistic if len(common)>1 else np.nan
            signs=sum(np.sign(a[k])==np.sign(b[k]) for k in common)/len(union) if union else np.nan
            lime_pairs.append({"position":int(pos),"run_a":i,"run_b":j,"shared":len(common),
                "union":len(union),"jaccard":len(common)/len(union) if union else np.nan,
                "kendall_tau_shared":tau,"sign_agreement_union":signs})
flat=[{k:v for k,v in r.items() if k not in ["features","descriptions"]} for r in lime_rows]
pd.DataFrame(flat).to_csv(RUN_DIR/"lime_local_agreement.csv",index=False)
pd.DataFrame(lime_pairs).to_csv(RUN_DIR/"lime_repeatability.csv",index=False)
write_json("lime_full_outputs.json",lime_rows)
write_json("lime_reference.json",{"positions":ref_positions.tolist(),"columns":lime_columns,"codebooks":codebooks})
print(pd.DataFrame(flat).groupby("run")[["fit_r2","local_abs_error","seconds"]].mean())

       fit_r2  local_abs_error   seconds
run                                     
0    0.528272         0.124682  0.230812
1    0.571490         0.122984  0.221270
2    0.557370         0.130542  0.222552
3    0.545429         0.128314  0.229605
4    0.579388         0.125505  0.232182
5    0.542683         0.124360  0.225837
6    0.599705         0.127114  0.220122
7    0.586200         0.125723  0.220669
8    0.584794         0.125296  0.230340
9    0.579557         0.125939  0.232168


## 10. Counterparty evidence and replayable integrated records
Only diagnostic cases receive full SHAP+LIME records in this version. All test cases receive primary predictions and rule coverage evaluation. Diagnostic records include supporting original row IDs for the selected rules. No pattern-ground-truth annotation enters these records. Attribution and domain inference remain separate.

In [11]:
def supporting_events(pos,hours=24):
    row=d.iloc[pos]; now=int(row.tsec); histmask=(d.tsec<now)&(d.tsec>now-hours*3600)&(d.src!=d.dst)
    account_mask=((d.dst==row.dst)|(d.src==row.src)|(d.dst==row.src))
    relevant=d.loc[histmask&account_mask,["row_id","src","dst","timestamp"]]
    return {"prior_row_ids":relevant.row_id.astype(int).tolist(),"current_row_id":int(row.row_id),
        "history_window_hours":hours,"strictly_earlier_timestamp":True}

records=[]; primary_model_hash=sha256_file(RUN_DIR/"model_xgb_history.joblib")
for i,pos in enumerate(diag):
    row=d.iloc[pos]; scores=primary_model.predict_proba(pre.transform(full.iloc[[pos]][primary["columns"]]))[0,1]
    top=np.argsort(np.abs(shap_diag[i]))[::-1][:CONFIG["top_k"]]
    records.append({"mode":MODE,"transaction_row_id":int(row.row_id),"event_time":row.timestamp.isoformat(),
        "generation_time":datetime.now(timezone.utc).isoformat(),"dataset_sha256":source_hash,
        "model":{"id":"xgb_history_seed42","sha256":primary_model_hash,"score":float(scores),
                 "threshold":primary["threshold"],"alert":bool(scores>=primary["threshold"]),"calibrated":False},
        "model_attribution":{"method":"TreeSHAP","output_scale":"raw_margin","feature_version":CONFIG["feature_version"],
            "baseline":float(np.asarray(explainer.expected_value).ravel()[0]),
            "full_vector_artifact":"shap_diagnostic.npz","diagnostic_vector_index":i,
            "top_features":[{"feature":encoded_names[j],"value":float(shap_diag[i,j])} for j in top]},
        "lime":lime_first[int(pos)],"domain_rule_evidence":rules_at(pos),"supporting_events":supporting_events(pos),
        "missing_context":["KYC/KYB","source of funds","customer risk classification","beneficial ownership","prior institutional cases"],
        "review":{"status":"PENDING_REVIEW","reviewer":None,"decision":None,"regulatory_filing":None},
        "conclusion_limit":"Research evidence, not proof of an offence or institutional compliance."})
write_json("integrated_records.json",records)
# Replay uses the saved raw count values and rule parameters, not the label.
replay=[]
for r in records:
    original=r["domain_rule_evidence"]; values=original[0]["observed_counts"]
    got=evaluate_rules([values["recipient_senders_inclusive"],values["sender_recipients_inclusive"],values["sender_senders_prior"]],
        original[0]["reason"]!="incomplete_global_observation_window",
        bool(d.loc[d.row_id==r["transaction_row_id"],"self_transfer"].iloc[0]),
        original[0]["counterparty_threshold"],original[0]["window_hours"])
    replay.append({"row_id":r["transaction_row_id"],"recorded_input_rule_replay_equal":got==original})
pd.DataFrame(replay).to_csv(RUN_DIR/"rule_record_replay.csv",index=False)
assert all(r["recorded_input_rule_replay_equal"] for r in replay)

## 11. A/B/C/D output ablation
This is an information/traceability comparison on identical cases and predictions, not a prediction-performance improvement. Required-field completeness is structural and partly by construction; report individual fields, not just the percentage. No claim about human comprehension is made.

Runtime below measures record assembly only. Detector, SHAP and LIME compute times are reported separately. Do not report assembly latency as end-to-end monitoring latency.

In [12]:
output_comparisons=[]
for r in records:
    for config in ["A","B","C","D"]:
        start=time.perf_counter()
        obj={"transaction_row_id":r["transaction_row_id"],"event_time":r["event_time"],"model":r["model"]}
        if config in ["B","C","D"]: obj["model_attribution"]=r["model_attribution"]
        if config in ["C","D"]: obj["domain_rule_evidence"]=r["domain_rule_evidence"]
        if config=="D":
            obj.update(dataset_sha256=r["dataset_sha256"],review=r["review"],
                missing_context=r["missing_context"],supporting_events=r["supporting_events"],lime=r["lime"])
        checks={"score_present":"score" in obj["model"],"attribution_present":"model_attribution" in obj,
            "rule_evidence_present":"domain_rule_evidence" in obj,"dataset_provenance_present":"dataset_sha256" in obj,
            "event_support_present":"supporting_events" in obj,"missing_context_present":"missing_context" in obj,
            "review_state_present":"review" in obj}
        elapsed=time.perf_counter()-start
        output_comparisons.append({"row_id":r["transaction_row_id"],"configuration":config,**checks,
            "structural_completeness":sum(checks.values())/len(checks),"assembly_seconds":elapsed,
            "serialized_bytes":len(json.dumps(plain(obj),allow_nan=False).encode())})
pd.DataFrame(output_comparisons).to_csv(RUN_DIR/"output_ablation.csv",index=False)

## 12. Exploratory subgroup performance — not demographic fairness
Currency-specific amount boundaries are learned from training rows. Unknown currencies receive their own unavailable-tier label. Sparse groups are flagged. There is no 0.80 demographic-fairness pass/fail rule and no demographic proxy assumption.

In [13]:
bounds={}
for currency,group in d.loc[train].groupby("pay_currency"):
    bounds[currency]=group.paid.quantile([.25,.75]).tolist()
def tier(row):
    if row.pay_currency not in bounds: return "UNSEEN_CURRENCY"
    lo,hi=bounds[row.pay_currency]
    return "LOW" if row.paid<=lo else "MEDIUM" if row.paid<=hi else "HIGH"
test_frame=d.loc[test].copy(); test_frame["tier"]=test_frame.apply(tier,axis=1)
subgroups=[]
for grouping,labels in [("payment_format",test_frame.payment_format),
                         ("currency_amount_tier",test_frame.pay_currency+"|"+test_frame.tier)]:
    for group in sorted(labels.unique()):
        mask=(labels==group).to_numpy()
        met=metrics(y[test][mask],primary_scores[mask],primary["threshold"])
        subgroups.append({"grouping":grouping,"group":group,**met,
            "sparse_positive_group":int(y[test][mask].sum())<CONFIG["sparse_positive_cutoff"],
            "interpretation":"Operational segment only; not demographic fairness."})
pd.DataFrame(subgroups).to_csv(RUN_DIR/"subgroup_results.csv",index=False)
write_json("training_amount_tiers.json",bounds)

## 13. Hash-linked audit log and controlled threat-model tests
Hashes detect edits, ordering changes and truncation **against a separately retained trusted checkpoint**. Anyone able to rewrite both the chain and checkpoint can evade this mechanism. Local notebook storage is not an independent trust anchor, access-control system or immutable production store.

Append-only simulated review events preserve prior events. These events are labelled SIMULATED, not evidence of real compliance-officer review. Original artefacts are never modified by the tests; tampering is performed on copies.

In [14]:
def canonical(x):
    return json.dumps(plain(x),sort_keys=True,separators=(",",":"),allow_nan=False).encode()
def make_chain(payloads):
    chain=[]; previous="0"*64
    for seq,payload in enumerate(payloads):
        body={"sequence":seq,"previous_hash":previous,"payload":payload}
        digest=hashlib.sha256(canonical(body)).hexdigest()
        chain.append({**body,"hash":digest}); previous=digest
    return chain,previous

def verify_chain(chain,checkpoint):
    previous="0"*64
    for seq,event in enumerate(chain):
        body={k:event[k] for k in ["sequence","previous_hash","payload"]}
        if event["sequence"]!=seq or event["previous_hash"]!=previous: return False
        if hashlib.sha256(canonical(body)).hexdigest()!=event["hash"]: return False
        previous=event["hash"]
    return previous==checkpoint

payloads=[{"event_type":"GENERATED_RECORD","record":r} for r in records]
if not payloads: raise ValueError("No diagnostic records; cannot integrity-test an empty chain.")
for state in ["REVIEWED","CLOSED","ESCALATED"]:
    payloads.append({"event_type":"SIMULATED_WORKFLOW_TRANSITION","transaction_row_id":records[0]["transaction_row_id"],
        "status":state,"actor":"TEST_ACTOR_NOT_A_REAL_REVIEWER","simulation":True})
chain,checkpoint=make_chain(payloads)
write_json("audit_chain.json",chain)
write_json("trusted_checkpoint_demo.json",{"checkpoint":checkpoint,
    "warning":"Demo colocates checkpoint with log. Production requires separate protected trust anchor."})
variants={"unaltered":(copy.deepcopy(chain),True)}
edited=copy.deepcopy(chain); edited[0]["payload"]["record"]["model"]["score"]=.123456
variants["edited_field"]=(edited,False)
deleted=copy.deepcopy(chain); del deleted[1]; variants["deleted_event"]=(deleted,False)
reordered=copy.deepcopy(chain); reordered[0],reordered[1]=reordered[1],reordered[0]; variants["reordered_events"]=(reordered,False)
variants["truncated_tail"]=(copy.deepcopy(chain[:-1]),False)
integrity=[]
for name,(candidate,expected) in variants.items():
    actual=verify_chain(candidate,checkpoint)
    integrity.append({"test":name,"expected_valid":expected,"actual_valid":actual,"passed":actual==expected})
assert all(x["passed"] for x in integrity)
pd.DataFrame(integrity).to_csv(RUN_DIR/"integrity_tests.csv",index=False)
# Explicitly demonstrate the limit of a compromised checkpoint.
rewritten=[x["payload"] for x in edited]
new_chain,new_checkpoint=make_chain(rewritten)
write_json("integrity_threat_limit.json",{"rewrite_chain_and_checkpoint_passes":verify_chain(new_chain,new_checkpoint),
    "meaning":"If both artefacts can be rewritten, this mechanism does not establish immutability."})
print(pd.DataFrame(integrity))

               test  expected_valid  actual_valid  passed
0         unaltered            True          True    True
1      edited_field           False         False    True
2     deleted_event           False         False    True
3  reordered_events           False         False    True
4    truncated_tail           False         False    True


## 14. Requirement-level assessment worksheet
Statuses are deliberately left for evidence review. Do not auto-convert a successful notebook execution into 'CBN compliant'. Replace PENDING_REVIEW only after inspecting the referenced artefacts. Exact source passages must be checked in the supplied CBN circular before final thesis submission.

In [15]:
assessment=[
    ["5.5(a)(v)","Auditable alert factors","shap_checks.csv; integrated_records.json","PENDING_REVIEW","No user comprehension study."],
    ["5.5(a)(iv-vi)","CDD/KYC/KYB and customer context","integrated_records.json","PARTIALLY_ADDRESSED","Institutional KYC and risk classifications unavailable."],
    ["5.5(b)(ii-iii)","Threshold and tuning documentation","configuration.json; validation_tuning.csv; model_registry.json","PENDING_REVIEW","No annual institutional review demonstrated."],
    ["5.5(b)(i)","Independent validation","rule_unit_tests.csv; classification_results.csv","UNTESTED","Developer testing is not independent validation."],
    ["5.9(a)(i-iv)","Immutable retrievable records","integrity_tests.csv; audit_chain.json","PARTIALLY_ADDRESSED","Local checkpoint not separately protected; no production immutability."],
    ["5.5(b)(v-vi)","Governance and timely human disposition","audit_chain.json","PARTIALLY_ADDRESSED","Workflow simulations are not staffed review or SLA evidence."]]
pd.DataFrame(assessment,columns=["CBN_section","expectation","artefact","status","boundary"]).to_csv(RUN_DIR/"CBN_assessment_worksheet.csv",index=False)
write_json("research_claim_boundaries.json",{
    "real_Nigerian_validation":False,"human_comprehension_tested":False,"demographic_fairness_tested":False,
    "independent_validation_completed":False,"production_immutability_demonstrated":False,
    "pattern_annotation_correspondence_tested":False,"all_alerts_explained":False,
    "full_CBN_compliance_certified":False,"smoke_results_eligible_for_dissertation":False})

## 15. Export environment, integrity manifest and a single run archive
This exports only generated results, models and diagnostic references, not the source dataset. Licence and institutional approval requirements must still be reviewed before sharing. Keep the original dataset separately so row IDs remain replayable.

Do not copy SMOKE metrics into the dissertation. Research results require a completed RESEARCH run and inspection of warnings, unit tests, local agreement, sparse subgroups and all failed or unavailable cases.

In [16]:
import importlib.metadata, shutil
versions={p:importlib.metadata.version(p) for p in ["numpy","pandas","scikit-learn","xgboost","shap","lime","scipy","joblib"]}
write_json("environment.json",{"python":sys.version,"platform":platform.platform(),"packages":versions,
    "mode":MODE,"run_id":RUN_ID,"cpu_count":os.cpu_count(),"configuration_sha256":sha256_file(RUN_DIR/"configuration.json")})
(RUN_DIR/"requirements_resolved.txt").write_text("\n".join(f"{k}=={v}" for k,v in versions.items())+"\n")
# Final evidence manifest excludes itself to avoid a self-referential digest.
manifest_files=[{"file":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in sorted(RUN_DIR.iterdir()) if p.is_file()]
write_json("artefact_manifest.json",manifest_files)
summary={"mode":MODE,"dataset":source_name,"accepted_rows":len(d),"diagnostic_cases":len(diag),
    "completed":True,"not_for_dissertation":MODE!="RESEARCH",
    "required_review":["convergence warnings","all test results","CBN worksheet","rule source authority",
        "no human study","unavailable customer context","pattern annotations not evaluated"],
    "deviations":CONFIG["deviations"]}
write_json("completion_summary.json",summary)
archive=shutil.make_archive(str(RUN_DIR),"zip",root_dir=RUN_DIR)
print(json.dumps(plain(summary),indent=2))
print("Archive:",archive)
# Optional in Colab: uncomment after successful completion.
# from google.colab import files
# files.download(archive)

{
  "mode": "RESEARCH",
  "dataset": "HI-Small_Trans.csv",
  "accepted_rows": 311724,
  "diagnostic_cases": 46,
  "completed": true,
  "not_for_dissertation": false,
  "required_review": [
    "convergence warnings",
    "all test results",
    "CBN worksheet",
    "rule source authority",
    "no human study",
    "unavailable customer context",
    "pattern annotations not evaluated"
  ],
  "deviations": []
}
Archive: /content/aml_runs/research_20261015T165721_873858Z.zip


## Interpretation checklist before Chapter Four
- Confirm RESEARCH mode and the correct source checksum.
- Inspect actual partitions, class distributions, quarantines and training warnings.
- Compare models on the same frozen test set; primary selection was validation-only.
- Keep seed variation separate from data-sampling uncertainty; no population confidence intervals are produced here.
- SHAP additivity is not universal fidelity. LIME R² describes a sampled local approximation.
- Rules identify candidate structures, not intent or validated laundering typologies.
- A/B/C/D completeness is a structural, partly by-construction measure, not a user benefit.
- Diagnostic samples are not population samples.
- Integrity tests demonstrate the stated threat model only.
- Pattern-annotation correspondence and independently adjudicated typology accuracy remain NOT EVALUATED.
- Record any protocol deviation before promoting exploratory reruns to primary findings.

### Bibliographic starting points
Altman et al. (2023), NeurIPS 36, 29851–29874. Nauta et al. (2023), ACM Computing Surveys, doi:10.1145/3583558. CBN (2026), *Baseline Standards for Automated AML Solutions*, circular BSD/DIR/PUB/LAB/019/002. NIST (2023), AI RMF 1.0, doi:10.6028/NIST.AI.100-1. Check chapter bibliographies against final publisher metadata. Code settings and rule thresholds in this notebook are researcher-defined, not source findings.